In [12]:
from openai import OpenAI
from dotenv import load_dotenv
import os
import json
import requests

load_dotenv()

client = OpenAI(
    api_key=os.getenv('GROQ_TOKEN'),
    base_url=os.getenv('BASE_URL_GROQ')
)

def get_response(messages:list[dict], **kwargs):
    response = client.chat.completions.create(
        model=os.getenv("GROQ_MODEL"),
        messages=messages,
        **kwargs
    )

    return response

In [16]:
# Часть 1
## Задача 1.1

text = '''Меня зовут Мария Соколова. Я Frontend-разработчик с 4 годами опыта.
Работаю в Москве, в офисе компании CloudTech. Специализируюсь на React и TypeScript.
Ищу удаленную работу с зарплатой от 200 000 рублей.'''


function_definition = [
    {
        'type':'function',
        'function': {
            'name':'extract_candidate_info',
            'description':'Функция используется для получения информации о кондидате из предоставленного текста',
            'parameters':{
                'type':'object',
                'properties': {
                    'name': {'type':'string', 'description':'полное ФИО кандидата'},
                    'job_title': {'type':'string', 'description':'желаемая должность'},
                    'years_experience':{'type':'string', 'description':'кол-во лет опыта работы'},
                    'location':{'type':'string', 'description':'текущий город работы'},
                    'skills':{'type':'string', 'description':'список ключевых навыков'}
                }
            }
        }
    }
]

messages = [
    {
        'role':'system',
        'content':'Ты ИИ асситент, твоя задача структурировать анкеты и и отзывы кандидатов в структурирован виде. Если каких то данных нет в переданной информации не выдумывай а заполни графу строкой "не указано"'
    },
    {
        'role':'user',
        'content':text
    }
]

response = get_response(messages=messages, tools = function_definition)

parsed_response = json.loads(response.choices[0].message.tool_calls[0].function.arguments)

parsed_response

{'job_title': 'Frontend-разработчик',
 'location': 'Москва',
 'name': 'Мария Соколова',
 'skills': 'React, TypeScript',
 'years_experience': '4 года'}

In [24]:
# Часть 2
## Задача 2.1

new = '''Компания Tesla объявила о запуске нового электромобиля Model Q в Берлине, Германия.
Презентация состоялась 15 ноября 2024 года на заводе Gigafactory Berlin.
Илон Маск лично представил модель, назвав её "революцией для массового рынка".
Стартовая цена составит $35,000. Первые поставки запланированы на март 2025 года.'''

function_definition = [
    {
        'type':'function',
        'function':{
            'name':'extract_event_info',
            'description':'Функция которая обрабатывает основную информацию',
            'parameters':{
                'type':'object',
                'properties':{
                    'company':{
                        'type':'string',
                        'description':'Наименование компании о которой информация'
                    },
                    'product':{
                        'type':'string',
                        'description':'Наименование продукта о которой информация'
                    },
                    'location':{
                        'type':'string',
                        'description':'Наименование места события'
                    },
                    'date':{
                        'type':'string',
                        'description':'Дата события'
                    }
                }
            }
        }
    },
    {
        'type':'function',
        'function':{
            'name':'extract_financial_info',
            'description':'Функция которая обрабатывает финансовую информацию',
            'parameters':{
                'type':'object',
                'properties':{
                    'price':{
                        'type':'string',
                        'description':'Стартовая цена'
                    },
                    'currency':{
                        'type':'string',
                        'description':'валюта'
                    },
                    'delivery_date':{
                        'type':'string',
                        'description':'дата первых поставок'
                    }
                }
            }
        }
    }
]

messages = [
    {
        'role':'system',
        'content':'Ты ИИ асситент обработки предоставленных новостей, приводящий текст в структурированные данные. Если какой либо информации нет в новостно статье не придумывай а заполни поле текстом "нет информации"'
    },
    {
        'role':'user',
        'content':new
    }
]

response = get_response(messages=messages, tools = function_definition)

print('=== ИНФОРМАЦИЯ О СОБЫТИИ ===')
print(response.choices[0].message.tool_calls[0].function.arguments)
print('\n=== ФИНАНСОВАЯ ИНФОРМАЦИЯ ===')
print(response.choices[0].message.tool_calls[1].function.arguments)
print(f'\n Количество вызванных функций: {len(response.choices[0].message.tool_calls)}')


=== ИНФОРМАЦИЯ О СОБЫТИИ ===
{"company":"Tesla","date":"15 ноября 2024","location":"Берлин, Германия","product":"Model Q"}

=== ФИНАНСОВАЯ ИНФОРМАЦИЯ ===
{"currency":"USD","delivery_date":"март 2025 года","price":"$35,000"}

 Количество вызванных функций: 2


In [40]:
# Часть 3
## Часть 3.1
messages = [
    {
        'role':'system',
        'content':'Ты ИИ асситент обработки предоставленных новостей, приводящий текст в структурированные данные. Если какой либо информации нет в новостно статье не придумывай а заполни поле текстом "нет информации"'
    },
    {
        'role':'user',
        'content':new
    }
]

response_1 = get_response(
    messages=messages, 
    tools = function_definition,
    tool_choice = 'auto'
    )

response_2 = get_response(
    messages=messages, 
    tools = function_definition,
    tool_choice = {
        'type':'function',
        'function':{'name':'extract_event_info'}
    }
    )

messages = [
    {
        'role':'system',
        'content':'Ты ИИ асситент обработки предоставленных новостей, приводящий текст в структурированные данные. Если какой либо информации нет в новостно статье не придумывай а заполни поле текстом "нет информации"'
    },
    {
        'role':'user',
        'content':'Стартовая цена составит $35,000. Первые поставки запланированы на март 2025 года.'
    }
]

response_3 = get_response(
    messages=messages, 
    tools = function_definition,
    tool_choice = {
        'type':'function',
        'function':{'name':'extract_financial_info'}
    }
    )

print('=== tool_choice = "auto" ===')
print(f'Вызванные функции: {[i.function.name for i in response_1.choices[0].message.tool_calls]}')

print('\n=== Принудительно вызвать только `extract_event_info` ===')
print(f'Вызванные функции: {[i.function.name for i in response_2.choices[0].message.tool_calls]}')

print('\n=== Принудительно вызвать только `extract_financial_info` ===')
print(f'Вызванные функции: {[i.function.name for i in response_3.choices[0].message.tool_calls]}')

=== tool_choice = "auto" ===
Вызванные функции: ['extract_event_info', 'extract_financial_info']

=== Принудительно вызвать только `extract_event_info` ===
Вызванные функции: ['extract_event_info']

=== Принудительно вызвать только `extract_financial_info` ===
Вызванные функции: ['extract_financial_info']


In [ ]:
## Задача 3.2
question = 'Какова годовая выручка компании Tesla?'
messages_no_sys = [
    {
        'role':'user',
        'content':new + f'\n Вопрос: {question}'
    }
]

response_no_sys = get_response(messages=messages_no_sys, tools = function_definition)

messages_with_sys = [
    {
        'role':'system',
        'content':'Ты ИИ асситент обработки предоставленных новостей, приводящий текст в структурированные данные. Если какой либо информации нет в новостно статье не придумывай а заполни поле текстом "нет информации". Также если тебе задали вопрос на который нет информации в предоставленной информации ответь сообщением "Нет информации чтобы ответить на Ваш вопрос"'
    },
    {
        'role':'user',
        'content':new + f'\n Вопрос: {question}'
    }
]

response_with_sys = get_response(messages=messages_with_sys, tools=function_definition)

for label, msgs in [("Без системного", messages_no_sys), ('С системным', messages_with_sys)]:
    response = get_response(messages=msgs, tools=function_definition)

    tool_calls = response.choices[0].message.tool_calls
    content = response.choices[0].message.content

    print(f"\n=== {label} сообщением ===")
    print(f"tool_calls: {tool_calls}")
    print(f"content: {content}")


=== Без системного сообщением ===
tool_calls: None
content: В предоставленной информации нет данных для расчета годовой выручки компании Tesla. Указана только стартовая цена нового электромобиля Model Q ($35 000) и дата первых поставок (март 2025 года), но объемы продаж, производственные мощности и другие факторы, влияющие на выручку, не указаны. Для точного ответа требуется обратиться к официальной финансовой отчетности Tesla за последний год.

<no_tool>
</no_tool>

=== С системным сообщением ===
tool_calls: None
content: Нет информации чтобы ответить на Ваш вопрос о годовой выручке компании Tesla.


В обоих случаях модель сообщила об отсутствии информации на поставленный вопрос, скорее всего это связанно с тем что современные модели стали более обученными и меньше галюционируют.

In [65]:
# Часть 4
## Задача 4.1

def search_books(query):
    response = requests.request(method='GET', url = f'https://openlibrary.org/search.json?q={query}&limit=5')

    return response.text

prompts = [
    '"Хочу почитать что-нибудь про космос и освоение других планет"',
    '"Посоветуй детективный роман, желательно классику"'
]

system_prompt = 'Ты ИИ библиотекарь. Основываясь на предпочтениях пользователя выдели одно ключевое слово или словосочетание для поиска соответствующих книг в бибилиотеке.Результат должен быть строго на английском языке. Если запрос превышает твои полномочия вежливо откажись отвечать и подскажи пользователю для чего ты создан.'

function_definition = [
    {
        'type':'function',
        'function':{
            'name':'get_book_recommendations',
            'description':'Функция которая выделяет одно ключевое слово или словосочетание из запроса пользователя',
            'parameters':{
                'type':'object',
                'properties':{
                    'search_query':{
                        'type':'string',
                        'description':'Ключевое слово или словосочетание'
                    }
                }
            }
        }
    }
]

for prompt in prompts:
    print(f'\nПользователь: {prompt}\n')
    messages = [
        {
            'role':'system',
            'content':system_prompt
        },
        {
            'role':'user',
            'content':prompt
        }
    ]
    response = get_response(messages=messages, tools=function_definition)

    if response.choices[0].finish_reason == 'tool_calls':
        keyword = json.loads(response.choices[0].message.tool_calls[0].function.arguments)

        print(f'Ключевое слово/словосочетание для поиска: {keyword['search_query']}\n')

        books_list = search_books(keyword['search_query'])
        books_dict = json.loads(books_list)

        if books_dict['docs']:

            for i, book in enumerate(books_dict['docs']):
                print(f'{i+1}. {book['title']}')
        
        else:
            print(f'По ключевому слову/словосочетанию: {keyword['search_query']} ничего не найдено!')
    else:
        print(response.choices[0].message.content)


Пользователь: "Хочу почитать что-нибудь про космос и освоение других планет"

Ключевое слово/словосочетание для поиска: space colonization

1. Space colonization
2. Space colonization
3. The Tempest
4. The Moon colonization. The Bagrov’s Space Colonization Strategy. Space Colonization Journal, Vol. 3, 2013.
5. Guards! Guards!

Пользователь: "Посоветуй детективный роман, желательно классику"

Ключевое слово/словосочетание для поиска: detective classic

1. Tom Sawyer, Detective
2. The Secret Garden
3. The Secret Agent
4. The Mysterious Affair at Styles
5. The Man Who Was Thursday


In [68]:
# Задача 5

prompts = [
    "Где мой заказ #12345?",
    "Расскажите про беспроводные наушники Sony",
    "Хочу вернуть заказ #67890, товар пришел бракованным",
]

function_definition = [
    {
        'type':'function',
        'function':{
            'name':'check_order_status',
            'description':'Проверка статуса заказа по номеру заказа',
            'parameters':{
                'type':'object',
                'properties':{
                    'order_id':{
                        'type':'string',
                        'description':'Номер заказа'
                    }
                }
            }
        }
    },
    {
        'type':'function',
        'function':{
            'name':'get_product_info',
            'description':'Получение информации о продукте по наименованию',
            'parameters':{
                'type':'object',
                'properties':{
                    'product_name':{
                        'type':'string',
                        'description':'Наименование продукта'
                    }
                }
            }
        }
    },
    {
        'type':'function',
        'function':{
            'name':'process_return_request',
            'description':'Процесс возврата товара, получение статуса и причин по номеру заказа',
            'parameters':{
                'type':'object',
                'properties':{
                    'order_id':{
                        'type':'string',
                        'description':'Номер заказа'
                    },
                    'reason':{
                        'type':'string',
                        'description':'Причина возврата'
                    }
                }
            }
        }
    }
]

system_prompt = '''Ты сотрудник поддержки интернет-магазина. В зависимости от запроса клиента ты направляешь обращения в определенные API в виде структурированных данных. Если каких то данных не будет указанно не придумывай и в ответ напиши сообщение "Недостаточно информации по вашему обращению, пожалуйста укажите (Тут запроси требуемую информацию для корректности обращения)"'''

messages = [
    {
        'role':'system',
        'content':system_prompt
    }
]

for prompt in prompts:
    print(f'Запрос: {prompt}')
    messages.append({'role':'user', 'content':prompt})
    response = get_response(messages=messages, tools=function_definition)
    print(f'Функция: {response.choices[0].message.tool_calls[0].function.name}')
    print(f'Параметры: {response.choices[0].message.tool_calls[0].function.arguments}')
    print()

Запрос: Где мой заказ #12345?
Функция: check_order_status
Параметры: {"order_id":"12345"}

Запрос: Расскажите про беспроводные наушники Sony
Функция: get_product_info
Параметры: {"product_name":"беспроводные наушники Sony"}

Запрос: Хочу вернуть заказ #67890, товар пришел бракованным
Функция: process_return_request
Параметры: {"order_id":"67890","reason":"бракованный товар"}

